# 07 - Information Loss with Text Preserved Information (TPI)

## Objective

Notebook 05 measures semantic similarity between the original and anonymized
texts using `sentence-transformers/all-mpnet-base-v2`.

Notebook 06 evaluates their logical relationship using
`cross-encoder/nli-deberta-v3-base`.

This notebook introduces a third and complementary dimension:

> **How much information contained in the original text is lost after anonymization?**

We use the **Text Preserved Information (TPI)** framework proposed by
Manzanares-Salor et al. (2026).

The central idea is to estimate the information content of the terms in the
original text with a pretrained masked language model.

For a term $t$:

$$
IC(t) = -\log P(t)
$$

where $P(t)$ is its contextual probability.

The total information content of a text is:

$$
TIC(D) = \sum_{t\in D} IC(t)
$$

If $M$ denotes the terms or spans from the original text whose information is
affected by anonymization, the corresponding TPI measure is:

$$
TPI = 1 -
\frac{\sum_{t\in M} IC(t)}
{TIC(W_o)}
$$

where $W_o$ is the original text.

This notebook therefore focuses on **information content and information loss**,
rather than semantic similarity or NLI.

In [1]:
import math
import gc
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

ROOT = Path.cwd().parent
module_path = ROOT / "src" / "content" / "tpi.py"

spec = importlib.util.spec_from_file_location(
    "content_tpi", module_path
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

estimate_all_term_ic = module.estimate_all_term_ic
information_content = module.information_content
masked_term_probabilities = module.masked_term_probabilities
term_indices = module.term_indices
total_information_content = module.total_information_content
tpi_from_spans = module.tpi_from_spans
whitespace_terms = module.whitespace_terms
term_subtokens = module.term_subtokens

/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "bert-base-uncased"
N = 6

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = model.to(device)
model.eval()

print("Model:", MODEL_NAME)
print("Device:", device)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 202/202 [00:00<00:00, 8684.48it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: bert-base-uncased
Device: cpu


## 2. Why a separate information-loss measure?

The results from **Notebook 6** showed that both semantic similarity and NLI
provide useful but different information about content preservation.

Our **`sentence-transformers/all-mpnet-base-v2`** model measures the
**overall semantic similarity** between the original and anonymized texts.
However, a high similarity score does not necessarily mean that all
information from the original text has been preserved.

For example:

**Original**

> I moved to Geneva in 2022 and I have two brothers.

**Anonymized**

> I moved to Switzerland and I have two brothers.

The two texts remain semantically close, so `all-mpnet-base-v2` can produce a
relatively high similarity score. However, specific information has changed:
the exact location (`Geneva`) has been generalized to `Switzerland`, while the
year (`2022`) is no longer present.

The `cross-encoder/nli-deberta-v3-base` model provides another perspective by
evaluating the **logical relationship** between the texts. As observed in
Notebook 6, however, NLI is difficult to interpret for information removal or
generalization. The absence of information does not necessarily correspond to
a contradiction.

For example:

**Original**

> I moved to Geneva in 2022 and I have two brothers.

**Anonymized**

> I moved to Geneva.

The anonymized text does not explicitly contradict the original statement. It
simply contains less specific information. Depending on the formulation, the
NLI model won't classify this type of transformation as `contradiction`, hence doesn't account for information lost.

This motivates a third evaluation perspective:
- **TPI** → Total amount of information content preserved

TPI therefore complements the two evaluation signals from Notebook 6 by
explicitly focusing on **information content and information loss**.

## 3. Information content

The TPI framework uses the classical information-content definition:

$$
IC(t) = -\log P(t)
$$

where:

- $t$ is a term;
- $P(t)$ is the probability assigned to that term given its context.

A highly predictable term has low information content, while a less
predictable term has higher information content.

For example, a common function word is generally more predictable than a
specific entity such as a person's name, a location, or a date.

The probability is estimated with a pretrained **masked language model (MLM)**.
This makes the information content contextual: the same term can receive
different values in different sentences.

## 4. Contextual probability

To estimate $P(t)$, the target term is replaced by `[MASK]` while the rest of
the sentence is kept as context.

For example:

> I moved to `[MASK]`.

The masked language model assigns probabilities to possible tokens at the
masked position. The probability assigned to the original term is then used
to compute:

$$
IC(t) = -\log P(t)
$$

The TPI implementation uses a pretrained BERT masked language model as the
information-content estimator. It is not fine-tuned on the anonymization
benchmark.

## 5. Terms and subword tokens

The MLM operates on **subword tokens**, whereas the information we want to
evaluate is expressed in linguistic terms or spans.

A term can therefore correspond to one or several WordPiece tokens.

When a term contains several subword tokens, we mask all of its subword
positions and use the **minimum probability across its subword components** as
the term probability:

$$
P(t) = \min_j P(t_j)
$$

This ensures that a multi-subtoken term is evaluated through all of its
components.

In [3]:
example = "I moved to Geneva in 2022."

terms = whitespace_terms(example)
print(terms)

pd.DataFrame([
    {
        "term": item["term"],
        "subtokens": term_subtokens(item["term"], tokenizer),
    }
    for item in terms
])

[{'term': 'I', 'start': 0, 'end': 1}, {'term': 'moved', 'start': 2, 'end': 7}, {'term': 'to', 'start': 8, 'end': 10}, {'term': 'Geneva', 'start': 11, 'end': 17}, {'term': 'in', 'start': 18, 'end': 20}, {'term': '2022', 'start': 21, 'end': 25}]


,term,subtokens
0,I,[i]
1,moved,[moved]
2,to,[to]
3,Geneva,[geneva]
4,in,[in]
5,2022,"[202, ##2]"


## 6. Masking a term in context

For a target term, we replace its character span by the required number of
`[MASK]` tokens.

For a single-token term:

> I moved to Geneva.

becomes:

> I moved to `[MASK]`.

For a multi-token term, several mask positions are inserted. The MLM then
provides a probability for each original subtoken at its corresponding masked
position.

In [4]:
text = "I moved to [MASK] in 2022."

inputs = tokenizer(text, return_tensors="pt").to(device)
print(inputs)
with torch.no_grad():
    output = model(**inputs)

logits = output.logits[0]

print("Shape:", logits.shape)

# Find the [MASK] position
mask_position = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()

print("Mask position:", mask_position)

# Look at the 5 tokens with the highest scores
top_logits, top_ids = torch.topk(logits[mask_position], 10)

for logit, token_id in zip(top_logits, top_ids):
    token = tokenizer.convert_ids_to_tokens(token_id.item())
    print(f"{token:10s} → logit = {logit.item():.3f}")

{'input_ids': tensor([[  101,  1045,  2333,  2000,   103,  1999, 16798,  2475,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Shape: torch.Size([10, 30522])
Mask position: 4
london     → logit = 8.612
australia  → logit = 8.063
california → logit = 7.705
england    → logit = 7.506
canada     → logit = 7.453
paris      → logit = 7.270
toronto    → logit = 7.154
france     → logit = 7.077
sydney     → logit = 7.026
chicago    → logit = 6.931


In [5]:
text = "I moved to Geneva in 2022."
terms = whitespace_terms(text)

result = masked_term_probabilities(
    text,
    [3, 4],
    tokenizer,
    model,
    device,
)

result

{3: {'term': 'Geneva',
  'subtokens': ['geneva'],
  'subtoken_probabilities': [0.0008418204379267991],
  'probability': 0.0008418204379267991},
 4: {'term': 'in',
  'subtokens': ['in'],
  'subtoken_probabilities': [0.7942429184913635],
  'probability': 0.7942429184913635}}

## 7. Information content of a term

Once the contextual probability has been estimated:

$$
IC(t) = -\log P(t)
$$

A lower probability therefore corresponds to a higher information-content
value.

In [6]:
geneva_result = result[3]

print("Term:", geneva_result["term"])
print("P(term | context):", geneva_result["probability"])
print("IC(term):", information_content(geneva_result["probability"]))

Term: Geneva
P(term | context): 0.0008418204379267991
IC(term): 7.079943823083377


## 8. Context-masking strategies

Computing a separate MLM prediction for every term can be expensive.

The paper therefore considers two strategies:

### Sentence-Term Alternating (S-TA)

One term is masked at a time while the other terms remain visible.

This gives the most direct contextual estimate, but requires many MLM
forward passes.

### N-Term Alternating (N-TA)

Several terms are masked simultaneously. The process is repeated over
different groups so that every term is eventually evaluated.

The paper evaluates values such as:

$$
N=2,\quad N=4,\quad N=6.
$$

The authors report that **N=6** provides a close approximation to S-TA while
substantially reducing the computational cost.

We therefore use:

$$
\boxed{N=6}
$$

as the default setting below.

In [7]:
example_text = "I moved to Geneva in 2022 and I have two brothers."

ic_table = estimate_all_term_ic(
    text=example_text,
    tokenizer=tokenizer,
    model=model,
    device=device,
    strategy="N-TA",
    N=6,
)

ic_table2 = estimate_all_term_ic(
    text=example_text,
    tokenizer=tokenizer,
    model=model,
    device=device,
    strategy="S-TA",
)

print(ic_table)
print(ic_table2)

    index      term   probability         IC
0       0         I  7.041229e-01   0.350802
1       1     moved  7.564459e-01   0.279124
2       2        to  9.885169e-01   0.011550
3       3    Geneva  7.114778e-04   7.248166
4       4        in  9.587823e-01   0.042091
5       5      2022  1.679999e-07  15.599302
6       6       and  3.425890e-03   5.676394
7       7         I  8.232490e-01   0.194497
8       8      have  9.194420e-01   0.083988
9       9       two  3.769791e-01   0.975565
10     10  brothers  2.228662e-03   6.106354
    index      term   probability         IC
0       0         I  7.385346e-01   0.303087
1       1     moved  8.004031e-01   0.222640
2       2        to  9.804289e-01   0.019765
3       3    Geneva  5.651301e-04   7.478455
4       4        in  9.604553e-01   0.040348
5       5      2022  1.679999e-07  15.599302
6       6       and  1.124799e-02   4.487566
7       7         I  8.492867e-01   0.163358
8       8      have  9.404652e-01   0.061381
9       9 

## 9. Total Information Content

For an original text $W_o$, the total information content is:

$$
TIC(W_o)
=
\sum_{t\in W_o} IC(t)
$$

The denominator represents the total amount of information estimated to be
present in the original text.

Conceptually:

$$
W_o
\rightarrow
\{t_1,\ldots,t_n\}
\rightarrow
\{IC(t_1),\ldots,IC(t_n)\}
\rightarrow
TIC(W_o)
$$

In [8]:
tic_original = total_information_content(ic_table)
print("TIC(original):", tic_original)

TIC(original): 36.56783434739532


## 10. What counts as information loss?

A key issue for our application is that anonymization does not always replace
one term with another.

Information can be affected through several types of transformation:

- **replacement**: `Geneva` → `Switzerland`
- **generalization**: `Geneva` → `Switzerland`
- **deletion**: `since 2022` → removed
- **partial deletion**: part of a phrase is removed
- **multiple modifications**: several spans are replaced or removed

The important consequence is that the metric must consider information that
has disappeared **even when no replacement term exists**.

## 11. Identifying affected spans

TPI needs to know which information from the original text was protected,
modified, or removed.

This information should ideally come from the anonymization process itself,
for example through character offsets or replacement metadata.

The metric does **not** need to infer the affected span from semantic
similarity or NLI.

For example:

**Original**

> I live in Geneva since 2022.

**Anonymized**

> I live in Geneva.

The information associated with `Geneva` is still present. The affected span
is:

$$
M = \{\text{since 2022}\}
$$

In contrast:

**Original**

> I live in Lausanne.

**Anonymized**

> I live in Geneva.

The affected span is:

$$
M = \{\text{Lausanne}\}
$$

Thus, both **replacement** and **deletion** can be represented as affected
spans in the original text.

## 12. Example: deletion of information

Consider:

**Original**

> I live in Geneva since 2022.

**Anonymized**

> I live in Geneva.

Here, the location is preserved but the temporal information is removed.

The affected information is therefore the span `since 2022`.

For this controlled example, we identify the affected terms manually. In a
real anonymization pipeline, these spans should come from the anonymization
output or from an alignment procedure.

In [9]:
original = "I live in Geneva since 2022."
anonymized = "I live in Geneva."

terms = whitespace_terms(original)
pd.DataFrame(terms)

,term,start,end
0,I,0,1
1,live,2,6
2,in,7,9
3,Geneva,10,16
4,since,17,22
5,2022,23,27


In [10]:
affected_term_indices = [
    i for i, x in enumerate(terms)
    if x["term"].lower() in {"since", "2022"}
]

ic_original = estimate_all_term_ic(
    text=original,
    tokenizer=tokenizer,
    model=model,
    device=device,
    strategy="N-TA",
    N=6,
)

deletion_result = tpi_from_spans(
    ic_original,
    affected_term_indices
)

print("Affected terms:")
display(deletion_result["affected_terms"])

print("TIC(original):", deletion_result["TIC_original"])
print("Information lost:", deletion_result["information_lost"])
print("TPI:", deletion_result["TPI"])

Affected terms:


,index,term,probability,IC
4,4,since,0.024859,3.694553
5,5,2022,0.000003,12.632096


TIC(original): 27.483468562363647
Information lost: 16.32664936914293
TPI: 0.4059465481187139


## 13. Example: replacement of information

Now consider:

**Original**

> I live in Lausanne.

**Anonymized**

> I live in Geneva.

The original term `Lausanne` is modified, so its information content is
counted as affected.

TPI therefore captures the information removed from the original text. It
does **not** attempt to determine whether `Geneva` is semantically similar to
`Lausanne`.

This is an important distinction from a replacement-aware metric such as
**Text Preserved Similarity (TPS)**: TPI focuses on the amount of information
associated with affected original terms, while TPS additionally evaluates the
similarity between original and replacement content.

In [11]:
original = "I live in Geneva since 2022."
anonymized = "I live in Geneva for 4 years."

terms = whitespace_terms(original)

# Estimate IC for all terms in the original text
ic_original = estimate_all_term_ic(
    text=original,
    tokenizer=tokenizer,
    model=model,
    device=device,
    strategy="N-TA",
    N=6,
)

# Find the terms that were changed
affected_term_indices = [
    i for i, x in enumerate(terms)
    if x["term"].lower() in {"since", "2022"}
]

replacement_result = tpi_from_spans(
    ic_original,
    affected_term_indices
)

print("Affected terms:")
display(replacement_result["affected_terms"])

print("TIC(original):", replacement_result["TIC_original"])
print("Information lost:", replacement_result["information_lost"])
print("TPI:", replacement_result["TPI"])

Affected terms:


,index,term,probability,IC
4,4,since,0.024859,3.694553
5,5,2022,0.000003,12.632096


TIC(original): 27.483468562363647
Information lost: 16.32664936914293
TPI: 0.4059465481187139


## 14. Interpretation

TPI is the amount of preserved information after anonymization. In other words, TPI can be interpreted as the proportion of the original text's estimated
information content that is **not associated with affected terms**. 

A value close to 1 means that only a small amount of the original information
content is associated with modified or removed information.

A lower value means that a larger fraction of the original information content
has been affected.

Importantly, TPI is **information-weighted**. Removing a highly informative
term can have a much larger effect than removing a predictable term.

TPI therefore goes beyond simply counting how many words changed.

## 15. Relationship with the previous notebooks

The three evaluation signals now answer different questions:

| Evaluation | Model / method | Main question |
|---|---|---|
| Semantic similarity | `sentence-transformers/all-mpnet-base-v2` | How semantically similar are the two texts? |
| Logical relation | `cross-encoder/nli-deberta-v3-base` | What is the logical relation between them? |
| Information preservation | TPI | How much of the original information content is affected? |

These measures should not be collapsed into a single score at this stage.

They provide complementary evidence about content preservation:

- **MPNet** captures overall semantic closeness.
- **NLI** captures logical relations such as entailment, neutral, and
  contradiction.
- **TPI** quantifies information content associated with removed or modified
  information.

## 16. Important limitation: identifying affected spans

TPI itself does not solve the alignment problem.

For replacement and deletion, we need to know which parts of the original
text were affected. This can be obtained from:

1. anonymization metadata or character offsets, when available;
2. a deterministic alignment between original and anonymized text;
3. a dedicated span-detection/alignment procedure.

This is particularly important for our project because an anonymization system
may replace a speaker-specific entity, generalize an entity, remove a date or
other attribute, delete an entire phrase, or modify several spans
simultaneously.

Therefore, **span identification is a preprocessing step for TPI**, not part
of the TPI score itself.

## References

- Manzanares-Salor, B., Sánchez, D., & Lison, P. (2026).
  *Unsupervised utility evaluation of text anonymization methods via neural
  language models*. Neural Networks.

- TAE (Text Anonymization Evaluation) framework, which implements TPI and
  related information-preservation metrics.

The implementation above follows the TPI framework conceptually, using
`bert-base-uncased` as the masked language model and **N=6** as the default
N-Term Alternating strategy.